In [1]:
import os
import cv2
import numpy as np
import pytesseract
from glob import glob
from fpdf import FPDF
from PIL import Image
from scipy.signal import find_peaks
from matplotlib import pyplot as plt

In [2]:
def create_dir(path):
    if not os.path.exists(path):
        os.makedirs(path)

In [3]:
def extract_pages(video_path, video_name, threshold, save_dir = "__pages__"):
    
    name      = os.path.splitext(os.path.basename(video_path))[0]
    save_path = os.path.join(save_dir, name)
    create_dir(save_path)

    cap = cv2.VideoCapture(video_path)

    prev_gray = None
    prev_img = None
    page_count = 0
    frame_idx = 0
    save_next_change = False

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        if prev_gray is None:
            cv2.imwrite(os.path.join(save_path, f"page_{page_count}.png"), frame)
            page_count += 1
            prev_gray = gray
            prev_img = frame
            continue

        diff = cv2.absdiff(prev_gray, gray)
        score = diff.mean()

        if score > threshold:

            if(save_next_change):
                cv2.imwrite(os.path.join(save_path, f"tpage_{page_count}.png"), prev_img)
                save_next_change = False
            else :
                save_next_change = True
            
            prev_img = frame
            page_count += 1
            prev_gray = gray
              
        frame_idx += 1

    cap.release()
    print(f"Extracted {page_count} pages")

In [4]:
def get_scores(video_path):
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print("Error: Could not open video.")
        return []

    prev = None
    scores = []
    count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        if prev is not None:
            scores.append(cv2.absdiff(prev, gray).mean())

        prev = gray
        count += 1

    cap.release()
    print(f"Processed {count} frames")
    return scores

In [5]:
videos_path = glob("videos/*")
video_name = None
stat_array = []

In [6]:
for video_path in videos_path:
    video_name = os.path.splitext(os.path.basename(video_path))[0]
    print(f"\nAnalyzing: {video_name}")
    print(f"the path is {video_path}")

    stat_array = get_scores(video_path)

    if len(stat_array) > 0 :

      mean = np.mean(stat_array)
      median = np.median(stat_array)
      std = np.std(stat_array)
      max_val = np.max(stat_array)

      mean_std_threshold = mean + 2 * std
      percentile_threshold = np.percentile(stat_array, 95)

      extract_pages(video_path, video_name, mean_std_threshold, save_dir = "__pages__/")



Analyzing: Chapter 6 Economic indicators explained
the path is videos\Chapter 6 Economic indicators explained.mp4
Processed 1834 frames
Extracted 125 pages


In [13]:
import pytesseract
import PIL.Image
import cv2
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet

In [16]:
# OCR config
myconfig = r'--oem 3 --psm 6'

# Directory where pages are saved
image_dir = "__pages__/Chapter 6 Economic indicators explained/"

# Output files
text_output_file = "output.txt"
pdf_output_file = "output.pdf"
audio_output_file = "output.mp3"

In [17]:
# -----------------------------
# 1. Extract text from all images
# -----------------------------
all_text = ""

image_files = sorted([
    f for f in os.listdir(image_dir)
    if f.endswith(".png")
])

for image_file in image_files:
    image_path = os.path.join(image_dir, image_file)

    img = PIL.Image.open(image_path)
    text = pytesseract.image_to_string(img, config=myconfig)

    all_text += text + "\n\n"

print("✅ OCR extraction complete")

✅ OCR extraction complete


In [18]:
# -----------------------------
# 2. Save to TEXT file
# -----------------------------
with open(text_output_file, "w", encoding="utf-8") as f:
    f.write(all_text)

print("✅ Text file saved")

✅ Text file saved


In [19]:
# -----------------------------
# 3. Save to PDF
# -----------------------------
doc = SimpleDocTemplate(pdf_output_file)
styles = getSampleStyleSheet()

content = []
for line in all_text.split("\n"):
    content.append(Paragraph(line, styles["Normal"]))
    content.append(Spacer(1, 10))

doc.build(content)

print("✅ PDF file saved")

✅ PDF file saved


In [20]:
all_text

"Chapter Sixteen\nEconomic Indicators Explained\nPrepare for bad times and you will only know good times.\nRobert Kiyosaki (1947-)\n\nIn this chapter we're going to dig into the fundamental news flow that shape the\ncurrency flows we see every day. Then move on to examine how these moves.\nare driven by the constant stream of economic indicators which are derived from\nthe underlying fundamental data.\nAnd in order to do so, we need to understand what we mean by an economic\nindicator, as well as explain the attributes and characteristics of economic indi-\ncators in general. The chapter will cover this in three areas.\nFirst, to look at how economic indicators can be grouped into three broad types.\nSecond, to consider how the importance and relevance of an economic indicator\nchanges, depending on where we are in the economic cycle.\nThird and finally, to explore their relationship to market expectations, once the\ndata has been released.\n\n\nMoreover, many of the numbers reported a

In [22]:
from gtts import gTTS

In [23]:
# -----------------------------
# 4. Convert text → Speech
# -----------------------------
tts = gTTS(text=all_text, lang='en')
tts.save(audio_output_file)

print("✅ Audio file saved")

gTTSError: Failed to connect. Probable cause: Unknown